In [223]:
# -*- coding: utf-8 -*-

# DO NOT CHANGE
import pandas as pd
import numpy as np

def get_order(structure):
    # structure: dictionary of structure
    #            key is variable and value is parents of a variable
    # return list of learning order of variables 
    # ex) ['A', 'R', 'E', 'S', 'T', 'O']
    answer=[]
    #initialize
            
    while len(answer)<len(structure):
        for i,v in structure.items():
            if len(v)==0 and i not in answer:
                answer.append(i)
            for j in v:
                if j not in answer:
                    break
            else:
                if i not in answer:
                    answer.append(i)
    return answer

def learn_parms(data,structure,var_order):
    # data: training data
    # structure: dictionary of structure
    # var_order: list of learning order of variables
    # return dictionary of trained parameters (key=variable, value=learned parameters)
    parms={}
    
    for v in var_order:
        # 테이블
        if len(structure[v])!=0:
            table = data[structure[v]].drop_duplicates().reset_index(drop=True,inplace=False,)
            table = table.apply(lambda row: ','.join(row.values.astype(str)), axis=1)
            column_name = ""
            for i in structure[v]:
                column_name+=i
                column_name+=","
            column_name = column_name[:-1]
            table =pd.DataFrame(table,columns=[column_name])
        else:
            table = pd.DataFrame(['np'],columns=['np'])
        for c in data[v].unique():
            table[c] = 0
        table = table.set_index(table.columns[0])
        # 데이터 넣기
        temp =data[structure[v]+[v]]
        for i in temp.index:
            index=""
            for j in temp.iloc[i].values[:-1]:
                index+=j
                index+=','
            index=index[:-1]
            if index!="":
                table[temp.iloc[i].values[-1]].loc[[index]]+=1
            else:
                table[temp.iloc[i].values[-1]].loc[['np']] +=1
        for i in table.index:
            table.loc[[i]]/= table.loc[i].sum()
        parms[v] = table
    return parms
                
def print_parms(var_order,parms):
    # var_order: list of learning order of variables
    # parms: dictionary of trained parameters (key=variable, value=learned parameters)
    # print the trained parameters for each variable
    for var in var_order:
        print('-------------------------')
        print('Variable Name=%s'%(var))
        #TODO: print the trained paramters
        print(parms[var])
    return
    
data=pd.read_csv('https://drive.google.com/uc?export=download&id=1taoE9WlUUN4IbzDzHv7mxk_xSj07f-Zt', sep=' ')
str1={'A':[],'S':[],'E':['A','S'],'O':['E'],'R':['E'],'T':['O','R']}
order1=get_order(str1)
parms1=learn_parms(data,str1,get_order(str1))
print('-----First Structure------')
print_parms(order1,parms1)
print('')

str2={'A':['E'],'S':['A','E'],'E':['O','R'],'O':['R','T'],'R':['T'],'T':[]}
order2=get_order(str2)
parms2=learn_parms(data,str2,get_order(str2))
print('-----Second Structure-----')
print_parms(order2,parms2)
print('')

-----First Structure------
-------------------------
Variable Name=A
    adult  young    old
np                     
np  0.472   0.32  0.208
-------------------------
Variable Name=S
        F      M
np              
np  0.402  0.598
-------------------------
Variable Name=E
             high       uni
A,S                        
adult,F  0.639175  0.360825
adult,M  0.719424  0.280576
young,F  0.538462  0.461538
young,M  0.810526  0.189474
old,F    0.846154  0.153846
old,M    0.892308  0.107692
-------------------------
Variable Name=O
           emp      self
E                       
high  0.980822  0.019178
uni   0.925926  0.074074
-------------------------
Variable Name=R
           big     small
E                       
high  0.717808  0.282192
uni   0.866667  0.133333
-------------------------
Variable Name=T
                 car     train     other
O,R                                     
emp,big     0.584699  0.215847  0.199454
emp,small   0.547009  0.376068  0.076923
self,small